# 03 · Execution Time Analysis

This notebook analyses execution time across languages and its relationship to energy.

**Units:** Time values are in **milliseconds (ms)** (converted from raw µs at load time).
Energy values are in **Joules (J)**.

**Key questions:**
- Which languages execute fastest?
- How does time correlate with CPU and memory energy?
- What is the Energy-Delay Product (EDP = CPU energy × time, in J·ms)?

**Methodology:** Rankings (time, EDP) and heatmaps use the **two-step mean** (equal
benchmark weight) from `results_clean_runs.csv` via `lang_means()`. Spearman correlation is
used for the energy↔time relationship (more robust than Pearson for right-skewed data),
and the paradigm comparison keeps the non-parametric Kruskal-Wallis / Mann-Whitney tests.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations

import plot_style as ps
ps.apply_style()

# Canonical constants — single source: plot_style.
COL_CPU_ENERGY, COL_MEM_ENERGY = ps.COL_CPU_ENERGY, ps.COL_MEM_ENERGY
COL_TIME                       = ps.COL_TIME
COL_CPU_CARBON, COL_MEM_CARBON = ps.COL_CPU_CARBON, ps.COL_MEM_CARBON
PARADIGM        = ps.PARADIGM
PARADIGM_COLORS = ps.PARADIGM_COLORS
PARADIGM_ORDER  = ps.PARADIGM_ORDER
MEANPROPS       = ps.MEANPROPS
ALPHA           = ps.ALPHA

OUTPUTS_DIR = Path('outputs'); OUTPUTS_DIR.mkdir(exist_ok=True)

# Single source of truth: per-run rows (df) + per-cell means with EDP (df_mean).
df      = ps.load_runs()
df_mean = ps.cell_means(df)

def lang_means(cols):
    """Per-language two-step mean (equal benchmark weight) for column(s) `cols`."""
    return ps.lang_means(df_mean, cols)

print(f"Runs: {df.shape} | Cell-means: {df_mean.shape} | "
      f"{df['language'].nunique()} languages \u00d7 {df['benchmark'].nunique()} benchmarks")
df_mean.head(3)

## 1. Execution Time by Language

Boxplots sorted by **mean** execution time (ms), with the mean marked (▲). Log scale is used
because time spans several orders of magnitude across languages and benchmarks. The box still
shows the median/quartiles as a distribution reference.

In [ ]:
time_mean = lang_means(COL_TIME)
lang_order_time = time_mean.sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(15, 6))
bp = ax.boxplot(
    [df[df['language'] == lang][COL_TIME].values for lang in lang_order_time],
    labels=lang_order_time, patch_artist=True, showmeans=True, meanprops=MEANPROPS,
    medianprops=dict(color='black', linewidth=2),
    flierprops=dict(marker='x', markerfacecolor='red', markersize=5, alpha=0.6),
)
for patch, lang in zip(bp['boxes'], lang_order_time):
    patch.set_facecolor(PARADIGM_COLORS[PARADIGM[lang]])
    patch.set_alpha(0.75)

ax.set_yscale('log')
ax.set_title('Execution Time by Language — log scale (sorted by mean; ▲ = mean)', fontsize=13)
ax.set_xlabel('Language')
ax.set_ylabel('Execution Time (ms, log scale)')
legend_handles = [mpatches.Patch(color=PARADIGM_COLORS[p], label=p, alpha=0.75)
                  for p in PARADIGM_ORDER]
ax.legend(handles=legend_handles, title='Paradigm', loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
ps.save_fig(fig, '03_time_by_language')
plt.show()

> **Takeaway:** execution time spans ~3 orders of magnitude (log scale); AOT native binaries are fastest, interpreted languages slowest.

### Execution Time per Paradigm Group

The same execution-time distribution as above, split by paradigm on a log scale to show within-group spread (mean ▲).

In [ ]:
# Per-paradigm split of the distribution (same per-run data as the all-language
# boxplot above, grouped by paradigm to show within-group spread; mean = ▲).
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)
for ax, paradigm in zip(axes, PARADIGM_ORDER):
    langs = [l for l in lang_order_time if PARADIGM[l] == paradigm]
    data  = [df[df['language'] == l][COL_TIME].values for l in langs]
    bp = ax.boxplot(data, labels=langs, patch_artist=True, showmeans=True, meanprops=MEANPROPS,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='x', markerfacecolor='red', markersize=5, alpha=0.6))
    for patch in bp['boxes']:
        patch.set_facecolor(PARADIGM_COLORS[paradigm])
        patch.set_alpha(0.75)
    ax.set_yscale('log')
    ax.set_title(paradigm)
    ax.set_ylabel('Time (ms, log)' if paradigm == 'AOT' else '')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
fig.suptitle('Execution Time per Paradigm Group — log scale (ms)', fontsize=13)
plt.tight_layout()
ps.save_fig(fig, '03_time_per_paradigm')
plt.show()

### Execution Time Ranking

Languages ranked by **mean** execution time (ms) — fastest at the top.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
time_rank = lang_means(COL_TIME).sort_values()   # order bars by the plotted metric
colors = [PARADIGM_COLORS[PARADIGM[l]] for l in time_rank.index]
bars = ax.barh(time_rank.index, time_rank.values, color=colors, alpha=0.85, edgecolor='white')

# value label at the end of each bar
x_max = time_rank.max()
for bar, val in zip(bars, time_rank.values):
    ax.text(bar.get_width() + x_max * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}', va='center', ha='left', fontsize=8, color='#333333')
ax.set_xlim(0, x_max * 1.15)

ax.set_title('Execution Time Ranking — mean across all benchmarks (ms)', fontsize=12)
ax.set_xlabel('Mean Execution Time (ms)')
ax.set_ylabel('Language')
ax.invert_yaxis()   # fastest (lowest) at the top
legend_handles = [mpatches.Patch(color=PARADIGM_COLORS[p], label=p, alpha=0.85)
                  for p in PARADIGM_ORDER]
ax.legend(handles=legend_handles, title='Paradigm', loc='upper right')
plt.tight_layout()
ps.save_fig(fig, '03_time_ranking')
plt.show()

> **Takeaway:** the time ranking closely mirrors the CPU-energy ranking — AOT native binaries are fastest, the interpreted languages slowest — confirming time and energy track together.

## 2. Time vs CPU Energy

Scatter plot of mean execution time (ms) vs mean CPU energy (J) per language (two-step mean).
A strong correlation is expected. The quadrants reveal interesting outliers:
- **Top-left**: fast but energy-hungry (parallel overhead?)
- **Bottom-right**: slow but energy-efficient

In [ ]:
agg_time = df_mean.groupby('language').agg(
    time_mean  = (COL_TIME, 'mean'),
    cpu_mean   = (COL_CPU_ENERGY, 'mean'),
    mem_mean   = (COL_MEM_ENERGY, 'mean'),
    paradigm   = ('paradigm', 'first'),
).reset_index()

r, p = stats.spearmanr(agg_time['time_mean'], agg_time['cpu_mean'])
print(f"Spearman r(time ms, CPU energy J) = {r:.4f}, p = {p:.4f}")

fig, ax = plt.subplots(figsize=(10, 7))
for paradigm in PARADIGM_ORDER:
    sub = agg_time[agg_time['paradigm'] == paradigm]
    ax.scatter(sub['time_mean'], sub['cpu_mean'],
               color=PARADIGM_COLORS[paradigm], label=paradigm, s=80, zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row['language'],
                    (row['time_mean'], row['cpu_mean']),
                    textcoords='offset points', xytext=(6, 3), fontsize=8)

ax.set_xlabel('Mean Execution Time (ms)')
ax.set_ylabel('Mean CPU Energy (J)')
ax.set_title(f'Execution Time (ms) vs CPU Energy (J) — Spearman r={r:.3f}, p={p:.3f}', fontsize=12)
ax.legend(title='Paradigm')
plt.tight_layout()
ps.save_fig(fig, '03_time_vs_cpu_energy')
plt.show()

> **Takeaway:** execution time and CPU energy are strongly rank-correlated (Spearman printed above) — slower languages are also more energy-hungry.

## 3. Time vs Memory Energy

Same analysis for Memory Energy (J). Memory energy tends to correlate less tightly with
time because DRAM power draw depends more on allocation patterns than execution duration.

In [ ]:
r_mem, p_mem = stats.spearmanr(agg_time['time_mean'], agg_time['mem_mean'])
print(f"Spearman r(time ms, Memory energy J) = {r_mem:.4f}, p = {p_mem:.4f}")

fig, ax = plt.subplots(figsize=(10, 7))
for paradigm in PARADIGM_ORDER:
    sub = agg_time[agg_time['paradigm'] == paradigm]
    ax.scatter(sub['time_mean'], sub['mem_mean'],
               color=PARADIGM_COLORS[paradigm], label=paradigm, s=80, zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row['language'],
                    (row['time_mean'], row['mem_mean']),
                    textcoords='offset points', xytext=(6, 3), fontsize=8)

ax.set_xlabel('Mean Execution Time (ms)')
ax.set_ylabel('Mean Memory Energy (J)')
ax.set_title(f'Execution Time (ms) vs Memory Energy (J) — Spearman r={r_mem:.3f}, p={p_mem:.3f}', fontsize=12)
ax.legend(title='Paradigm')
plt.tight_layout()
ps.save_fig(fig, '03_time_vs_mem_energy')
plt.show()

> **Takeaway:** memory energy correlates with time more weakly than CPU energy, since DRAM draw depends on allocation patterns more than wall-clock duration.

## Correlation Summary — Spearman ρ with Rea–Parker effect size

Spearman rank correlation between the three primary metrics, on the per-language
means (N = 18). Rank-based, so robust to the right-skewed distributions.
Following **Rea & Parker**, the modulus of each ρ is given a nominal effect-size
label:

| \|ρ\| | Classification |
|------|----------------|
| 0.00 – 0.10 | Negligible |
| 0.10 – 0.20 | Weak |
| 0.20 – 0.40 | Moderate |
| 0.40 – 0.60 | Relatively strong |
| 0.60 – 0.80 | Strong |
| 0.80 – 1.00 | Very strong |

In [ ]:
# Spearman rank correlations between the three primary metrics, on the
# per-language two-step means (N = 18); rank-based, robust to the right-skew.
lm = df_mean.groupby('language')[[COL_CPU_ENERGY, COL_MEM_ENERGY, COL_TIME]].mean()

def classify(r):
    """Rea & Parker nominal effect-size label for |Spearman rho|."""
    a = abs(r)
    if a < 0.10: return 'Negligible'
    if a < 0.20: return 'Weak'
    if a < 0.40: return 'Moderate'
    if a < 0.60: return 'Relatively strong'
    if a < 0.80: return 'Strong'
    return 'Very strong'

pairs = [
    ('CPU Energy × Execution Time',    COL_CPU_ENERGY, COL_TIME),
    ('CPU Energy × Memory Energy',     COL_CPU_ENERGY, COL_MEM_ENERGY),
    ('Memory Energy × Execution Time', COL_MEM_ENERGY, COL_TIME),
]
rows = []
for name, a, b in pairs:
    rho, p = stats.spearmanr(lm[a], lm[b])
    rows.append({'Correlation': name, 'Spearman ρ': round(rho, 3),
                 'p-value': p, 'N': len(lm), 'Classification': classify(rho)})
corr_table = pd.DataFrame(rows).set_index('Correlation')
corr_table.to_csv(OUTPUTS_DIR / 'spearman_correlations.csv')
print('Saved → outputs/spearman_correlations.csv')

# Display-formatted copy (strings) for the styled table figure (PNG + PDF).
disp = pd.DataFrame({
    'Spearman ρ':     corr_table['Spearman ρ'].map(lambda v: f'{v:.3f}'),
    'p-value':        corr_table['p-value'].map(lambda v: f'{v:.1e}'),
    'N':              corr_table['N'].astype(str),
    'Classification': corr_table['Classification'],
}, index=corr_table.index)
ps.styled_table_fig(
    disp,
    'Spearman Correlations between Primary Metrics (per-language means, N=18)',
    '03_spearman_correlation_table',
    highlight_col='Classification',
)
corr_table

## 4. Paradigm Speed Comparison

Kruskal-Wallis test on execution time across paradigm groups, followed by pairwise
Mann-Whitney U tests with Bonferroni correction.

In [ ]:
import matplotlib.ticker as mticker

# Violin distribution of execution time per paradigm (log scale — time spans
# several orders of magnitude). Mirrors the energy violin in notebook 02.
fig, ax = plt.subplots(figsize=(8, 6))
groups = [np.log10(df[df['paradigm'] == p][COL_TIME].values) for p in PARADIGM_ORDER]
parts = ax.violinplot(groups, positions=range(len(PARADIGM_ORDER)),
                      showmedians=True, showmeans=True)
for pc, p in zip(parts['bodies'], PARADIGM_ORDER):
    pc.set_facecolor(PARADIGM_COLORS[p])
    pc.set_alpha(0.7)
for key in ('cmedians', 'cmeans', 'cbars', 'cmins', 'cmaxes'):
    if key in parts:
        parts[key].set_edgecolor('#333333')
        parts[key].set_linewidth(1)

ax.set_xticks(range(len(PARADIGM_ORDER)))
ax.set_xticklabels(PARADIGM_ORDER)
# show log-spaced axis with readable millisecond labels
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{10 ** v:,.0f}'))
ax.set_ylabel('Execution Time (ms, log scale)')
ax.set_title('Execution Time Distribution by Execution Paradigm', fontsize=13)
plt.tight_layout()
ps.save_fig(fig, '03_time_violin_paradigm')
plt.show()

> **Takeaway:** the three paradigms occupy clearly separated time bands — AOT fastest, JIT in the middle, interpreted slowest — with the Kruskal-Wallis / Mann-Whitney tests below confirming the separation is significant.

In [ ]:
def rank_biserial(x, y):
    u, _ = stats.mannwhitneyu(x, y, alternative='two-sided')
    return 1 - (2 * u) / (len(x) * len(y))

groups = {p: df[df['paradigm'] == p][COL_TIME].values for p in PARADIGM_ORDER}
kw_stat, kw_p = stats.kruskal(*groups.values())
n_pairs = len(PARADIGM_ORDER) * (len(PARADIGM_ORDER) - 1) // 2

print(f"Kruskal-Wallis (Execution Time, ms): H={kw_stat:.3f}, p={kw_p:.4f}")
print("SIGNIFICANT" if kw_p < ALPHA else "Not significant")

if kw_p < ALPHA:
    print(f"\nPost-hoc (Bonferroni α={ALPHA/n_pairs:.4f}):")
    for p1, p2 in combinations(PARADIGM_ORDER, 2):
        u, p = stats.mannwhitneyu(groups[p1], groups[p2], alternative='two-sided')
        p_adj = min(p * n_pairs, 1.0)
        r = rank_biserial(groups[p1], groups[p2])
        sig = "✓" if p_adj < ALPHA else "✗"
        print(f"  {sig} {p1} vs {p2}: p_adj={p_adj:.4f}, r={r:.3f}")

## 5. Energy-Delay Product (EDP)

**EDP = (CPU Energy + Memory Energy) (J) × Execution Time (ms)** — unit: **J·ms**

EDP is a standard hardware metric penalising both slow and energy-hungry implementations.
Including memory energy captures the full energy cost of execution. Lower EDP is better.

In [ ]:
# EDP per cell = (CPU + Mem energy) × time, computed in the load cell on df_mean.
# Two-step mean: average the per-benchmark EDP with equal benchmark weight.
edp_rank = (lang_means('EDP')
              .sort_values()
              .reset_index())
edp_rank.columns = ['language', 'EDP_mean_Jms']
edp_rank['paradigm'] = edp_rank['language'].map(PARADIGM)

print("EDP Ranking — lower is better (unit: J·ms):")
print(edp_rank[['language', 'paradigm', 'EDP_mean_Jms']].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 7))
colors = [PARADIGM_COLORS[p] for p in edp_rank['paradigm']]
bars = ax.barh(edp_rank['language'], edp_rank['EDP_mean_Jms'], color=colors, alpha=0.85, edgecolor='white')

# value label at the end of each bar
x_max = edp_rank['EDP_mean_Jms'].max()
for bar, val in zip(bars, edp_rank['EDP_mean_Jms']):
    ax.text(bar.get_width() + x_max * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}', va='center', ha='left', fontsize=8, color='#333333')
ax.set_xlim(0, x_max * 1.22)

ax.set_title('Energy-Delay Product Ranking — (CPU + Mem) Energy × Time (J·ms, mean)', fontsize=12)
ax.set_xlabel('EDP (J·ms)')
ax.set_ylabel('Language')
ax.invert_yaxis()
legend_handles = [mpatches.Patch(color=PARADIGM_COLORS[p], label=p, alpha=0.85)
                  for p in PARADIGM_ORDER]
ax.legend(handles=legend_handles, title='Paradigm', loc='lower right')
plt.tight_layout()
ps.save_fig(fig, '03_edp_ranking')
plt.show()

> **Takeaway:** the EDP ranking (penalising both slow and energy-hungry runs) is led by the AOT compilers; Erlang's long regex-redux run pushes it to the bottom.

## 6. Benchmark-Level Time Heatmap

Mean execution time (ms) for each language × benchmark cell (the per-cell means stored in
`results_clean_runs.csv`). Reveals which benchmarks are the slowest and which languages suffer
most on specific workloads.

In [ ]:
pivot_time = df_mean.pivot(index='language', columns='benchmark', values=COL_TIME)
lang_sort = lang_means(COL_TIME).sort_values().index
pivot_time = pivot_time.loc[lang_sort]

fig, ax = plt.subplots(figsize=(13, 9))
sns.heatmap(pivot_time, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            linewidths=0.3, cbar_kws={'label': 'Time (ms)'})
ax.set_title('Execution Time Heatmap — mean (ms)', fontsize=12)
ax.set_xlabel('Benchmark')
ax.set_ylabel('Language')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
ps.save_fig(fig, '03_time_heatmap_benchmark')
plt.show()

> **Takeaway:** regex-redux and k-nucleotide are the slowest workloads, and the interpreted languages pay the largest time penalty on them.